In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import os


In [ ]:
model=YOLO("best.pt")


In [ ]:
IMAGE_FOLDER = "My-First-Project-1/train/images"


In [ ]:
feature_data=[]
print("Starting feature extraction using segmentation...")

In [ ]:
for img_name in os.listdir(IMAGE_FOLDER):

    img_path = os.path.join(IMAGE_FOLDER, img_name)

    results = model(img_path)
    r = results[0]

    if r.masks is None:
        continue

    image = cv2.imread(img_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Compute gradient ONCE per image
    gradient = cv2.Laplacian(gray, cv2.CV_64F)

    masks = r.masks.data.cpu().numpy()

    for mask in masks:

        # Resize mask to match original image size
        mask = cv2.resize(mask, (gray.shape[1], gray.shape[0]))
        mask = mask > 0.5   # Convert to boolean

        mango_pixels = gray[mask]

        if len(mango_pixels) == 0:
            continue

        mean_T = np.mean(mango_pixels)
        max_T = np.max(mango_pixels)
        std_T = np.std(mango_pixels)
        var_T = np.var(mango_pixels)

        threshold = mean_T + 1.5 * std_T
        hotspot_ratio = np.sum(mango_pixels > threshold) / len(mango_pixels)

        gradient_mean = np.mean(np.abs(gradient[mask]))

        feature_data.append([
            img_name,
            mean_T,
            max_T,
            std_T,
            var_T,
            hotspot_ratio,
            gradient_mean
        ])

In [ ]:
columns = [
    "image_name",
    "mean_T",
    "max_T",
    "std_T",
    "var_T",
    "hotspot_ratio",
    "gradient_mean"
]
df = pd.DataFrame(feature_data, columns=columns)

df.to_csv("segmentation_features.csv", index=False)

print("Done.")
print("Total mango samples:", len(df))

In [ ]:
import pandas as pd

df = pd.read_csv("segmentation_features.csv")

print(df.head())
print(df.describe())

In [ ]:
q1 = df["mean_T"].quantile(0.25)
q2 = df["mean_T"].quantile(0.50)
q3 = df["mean_T"].quantile(0.75)

def assign_R(x):
    if x <= q1:
        return 1
    elif x <= q2:
        return 2
    elif x <= q3:
        return 3
    else:
        return 4

df["R_label"] = df["mean_T"].apply(assign_R)

In [ ]:
df["Q_score"] = (
    df["hotspot_ratio"] * 0.6 +
    df["gradient_mean"] * 0.4
)

In [ ]:
q1 = df["Q_score"].quantile(0.25)
q2 = df["Q_score"].quantile(0.50)
q3 = df["Q_score"].quantile(0.75)

def assign_Q(x):
    if x <= q1:
        return 1
    elif x <= q2:
        return 2
    elif x <= q3:
        return 3
    else:
        return 4

df["Q_label"] = df["Q_score"].apply(assign_Q)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

features = ["mean_T", "max_T", "std_T", "var_T", "hotspot_ratio", "gradient_mean"]

X = df[features]

# Train R model
y_R = df["R_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y_R, test_size=0.2, random_state=42)

r_model = RandomForestClassifier(n_estimators=200)
r_model.fit(X_train, y_train)

print("R Accuracy:", r_model.score(X_test, y_test))

In [ ]:
import joblib

joblib.dump(r_model, "r_model.pkl")
